# 논문 재현형 엄격 베이스라인

이 노트북은 논문 **"A Case Study on the Data Mining-Based Prediction of Students' Performance for Effective and Sustainable E-Learning"**의 예측 변수 구성을 OULAD 코호트 파일에 맞춰 엄격하게 단순화한 베이스라인 실험이다.

이 실험은 최종 모델링이 아니라, 논문 재현 관점의 기준 성능을 확인하기 위한 베이스라인이다. 따라서 학습 과정과 직접 관련된 두 가지 특징만 사용한다.

- `active_days_until_cutoff`: OULAD에는 직접적인 로그인 수가 없으므로, 컷오프 시점까지 활동한 날짜 수를 access/login proxy로 사용한다.
- `total_click_until_cutoff_*_strategy`: 컷오프 시점까지의 클릭 수를 click count 특징으로 사용한다.

인구통계 특징, 평가/assessment 특징, module/presentation 식별자, activity_type별 클릭 특징, `id_student`, `final_result`, target 컬럼은 엄격 재현 베이스라인의 입력 특징에서 의도적으로 제외한다.

## 1. 실행 환경과 공통 설정

프로젝트 루트에서 실행되는 것을 전제로 경로를 설정한다. 필요한 폴더가 없으면 생성하되, 원본 데이터 파일은 수정하지 않는다.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "reports").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed and reports")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data dir exists: {DATA_DIR.exists()}")
print(f"Report table dir: {REPORT_TABLE_DIR}")

Project root: C:\Users\osca0\Dev\studyDataMining\teamproject
Processed data dir exists: True
Report table dir: C:\Users\osca0\Dev\studyDataMining\teamproject\reports\tables


## 2. 실험 대상 파일과 특징 정의

week 5, week 7, week 10 코호트 파일을 사용한다. VLE 중복 처리 방식의 영향을 비교하기 위해 max strategy와 sum strategy를 각각 평가한다.

엄격 재현 베이스라인의 입력 특징은 각 전략별로 정확히 두 개다. `code_module`, `code_presentation`, `id_student`, `final_result`, target 컬럼은 입력 행렬 `X`에 넣지 않는다.

In [2]:
cutoff_files = {
    "week5": DATA_DIR / "features_week5_cohort.csv",
    "week7": DATA_DIR / "features_week7_cohort.csv",
    "week10": DATA_DIR / "features_week10_cohort.csv",
}

feature_sets = {
    "max": [
        "total_click_until_cutoff_max_strategy",
        "active_days_until_cutoff",
    ],
    "sum": [
        "total_click_until_cutoff_sum_strategy",
        "active_days_until_cutoff",
    ],
}

target_name = "target_struggling"

for cutoff, path in cutoff_files.items():
    print(f"{cutoff}: {path} | exists={path.exists()}")

week5: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week5_cohort.csv | exists=True
week7: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week7_cohort.csv | exists=True
week10: C:\Users\osca0\Dev\studyDataMining\teamproject\data\processed\features_week10_cohort.csv | exists=True


## 3. 타깃 생성과 입력 검증 함수

주요 타깃은 학업 실패/성공 프레이밍에 맞춰 `target_struggling`으로 정의한다.

- `Withdrawn`, `Fail` -> 1, struggling 양성 클래스
- `Pass`, `Distinction` -> 0, non-struggling 음성 클래스

행을 조용히 제거하지 않기 위해, 필요한 컬럼 누락, 알 수 없는 `final_result`, 결측값, 무한값이 있으면 즉시 오류를 발생시킨다.

In [3]:
target_map = {
    "Withdrawn": 1,
    "Fail": 1,
    "Pass": 0,
    "Distinction": 0,
}


def load_and_validate_cohort(path: Path, cutoff: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing cohort file for {cutoff}: {path}")

    df = pd.read_csv(path)
    required_columns = {"final_result", "active_days_until_cutoff"}
    for columns in feature_sets.values():
        required_columns.update(columns)

    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise ValueError(f"{cutoff} is missing required columns: {missing_columns}")

    unknown_results = sorted(set(df["final_result"].dropna().unique()) - set(target_map.keys()))
    if unknown_results:
        raise ValueError(f"{cutoff} has unexpected final_result values: {unknown_results}")

    if df["final_result"].isna().any():
        raise ValueError(f"{cutoff} has missing final_result values")

    df = df.copy()
    df[target_name] = df["final_result"].map(target_map).astype(int)

    print(f"\n[{cutoff}] rows: {len(df):,}")
    print("final_result distribution:")
    print(df["final_result"].value_counts().sort_index())
    print("target_struggling distribution:")
    print(df[target_name].value_counts().sort_index())

    return df


def build_xy(df: pd.DataFrame, features: list[str], cutoff: str, strategy: str):
    missing_values = df[features].isna().sum()
    if missing_values.any():
        raise ValueError(
            f"{cutoff}/{strategy} has missing feature values: "
            f"{missing_values[missing_values > 0].to_dict()}"
        )

    X = df.loc[:, features].copy()
    y = df[target_name].copy()

    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(f"{cutoff}/{strategy} has non-finite feature values")

    return X, y

## 4. 모델과 교차검증 설정

논문 재현형 기준 성능 비교를 위해 Decision Tree, Gaussian Naive Bayes, Random Forest, SVC, KNN을 동일한 5-fold StratifiedKFold로 평가한다.

양성 클래스는 struggling, 즉 `target_struggling = 1`이다. precision, recall, f1은 모두 struggling 클래스를 기준으로 계산한다.

In [4]:
models = {
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    ),
    "SVC": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", random_state=RANDOM_STATE)),
        ]
    ),
    "KNeighborsClassifier": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
}

print("Models:", list(models.keys()))
print("Scoring:", list(scoring.keys()))

Models: ['DecisionTreeClassifier', 'GaussianNB', 'RandomForestClassifier', 'SVC', 'KNeighborsClassifier']
Scoring: ['precision', 'recall', 'f1', 'roc_auc', 'accuracy']


## 5. 컷오프와 VLE 중복 전략별 평가

각 cutoff와 VLE 중복 처리 전략별로 정확히 두 개의 특징만 사용한다. 이 단계는 additional feature engineering을 하지 않으며, 이미 생성된 코호트 파일의 두 learning-process feature만 입력으로 사용한다.

In [7]:
rows = []

for cutoff, path in cutoff_files.items():
    df = load_and_validate_cohort(path, cutoff)

    for strategy, features in feature_sets.items():
        X, y = build_xy(df, features, cutoff, strategy)
        print(f"\nEvaluating {cutoff}/{strategy} with features: {features}")

        for model_name, model in models.items():
            scores = cross_validate(
                clone(model),
                X,
                y,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
            )

            result = {
                "cutoff": cutoff,
                "strategy": strategy,
                "model": model_name,
                "n_rows": len(df),
                "positive_class": "struggling",
                "features": ",".join(features),
            }

            for metric in scoring:
                values = scores[f"test_{metric}"]
                result[f"{metric}_mean"] = values.mean()
                result[f"{metric}_std"] = values.std(ddof=1)

            rows.append(result)
            print(
                f"  {model_name}: "
                f"f1={result['f1_mean']:.4f}, "
                f"recall={result['recall_mean']:.4f}, "
                f"roc_auc={result['roc_auc_mean']:.4f}, "
                f"accuracy={result['accuracy_mean']:.4f}"
            )

results = pd.DataFrame(rows)
results


[week5] rows: 27,244


final_result distribution:
final_result
Distinction     3024
Fail            7044
Pass           12361
Withdrawn       4815
Name: count, dtype: int64
target_struggling distribution:
target_struggling
0    15385
1    11859
Name: count, dtype: int64



Evaluating week5/max with features: ['total_click_until_cutoff_max_strategy', 'active_days_until_cutoff']


  DecisionTreeClassifier: f1=0.4884, recall=0.4718, roc_auc=0.5684, accuracy=0.5698


  GaussianNB: f1=0.6209, recall=0.7633, roc_auc=0.6730, accuracy=0.5942


...

  RandomForestClassifier: f1=0.5096, recall=0.5053, roc_auc=0.6227, accuracy=0.6014


  SVC: f1=0.5163, recall=0.4350, roc_auc=0.6818, accuracy=0.6658
  KNeighborsClassifier: f1=0.5075, recall=0.4758, roc_auc=0.6377, accuracy=0.6216


,cutoff,strategy,model,n_rows,positive_class,features,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std
0,week5,max,DecisionTreeClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.506331,0.005455,0.471794,0.009029,0.488409,0.005643,0.568405,0.005441,0.569814,0.004464
1,week5,max,GaussianNB,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.523263,0.004299,0.763301,0.005482,0.620877,0.003672,0.672957,0.003109,0.594223,0.005226
2,week5,max,RandomForestClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.508454,0.005570,0.525002,0.003861,0.516581,0.003741,0.595677,0.006471,0.572273,0.005008
3,week5,max,SVC,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.604104,0.005402,0.464120,0.009698,0.524876,0.005905,0.658852,0.003176,0.634305,0.002962
4,week5,max,KNeighborsClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.528010,0.010540,0.475166,0.010483,0.500191,0.010403,0.606032,0.007638,0.586661,0.008317
5,week5,sum,DecisionTreeClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.502443,0.008023,0.467663,0.008370,0.484381,0.006201,0.564044,0.008037,0.566620,0.006482
6,week5,sum,GaussianNB,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.521574,0.004439,0.771312,0.006357,0.622311,0.004280,0.673505,0.003113,0.592461,0.005516
7,week5,sum,RandomForestClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.509510,0.006247,0.526520,0.004350,0.517860,0.004480,0.591672,0.005116,0.573227,0.005525
8,week5,sum,SVC,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.604740,0.004937,0.468842,0.009892,0.528124,0.005821,0.658785,0.002916,0.635369,0.002643
9,week5,sum,KNeighborsClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.531268,0.003876,0.482840,0.006847,0.505882,0.004856,0.606051,0.003411,0.589451,0.003040


  KNeighborsClassifier: f1=0.5002, recall=0.4752, roc_auc=0.6060, accuracy=0.5867

Evaluating week5/sum with features: ['total_click_until_cutoff_sum_strategy', 'active_days_until_cutoff']
  DecisionTreeClassifier: f1=0.4844, recall=0.4677, roc_auc=0.5640, accuracy=0.5666
  GaussianNB: f1=0.6223, recall=0.7713, roc_auc=0.6735, accuracy=0.5925
  RandomForestClassifier: f1=0.5179, recall=0.5265, roc_auc=0.5917, accuracy=0.5732
  SVC: f1=0.5281, recall=0.4688, roc_auc=0.6588, accuracy=0.6354
  KNeighborsClassifier: f1=0.5059, recall=0.4828, roc_auc=0.6061, accuracy=0.5895

[week7] rows: 26,776
final_result distribution:
final_result
Distinction     3024
Fail            7044
Pass           12361
Withdrawn       4347
Name: count, dtype: int64
target_struggling distribution:
target_struggling
0    15385
1    11391
Name: count, dtype: int64

Evaluating week7/max with features: ['total_click_until_cutoff_max_strategy', 'active_days_until_cutoff']
  DecisionTreeClassifier: f1=0.4872, recall=0.4

,cutoff,strategy,model,n_rows,positive_class,features,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std
0,week5,max,DecisionTreeClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.506331,0.005455,0.471794,0.009029,0.488409,0.005643,0.568405,0.005441,0.569814,0.004464
1,week5,max,GaussianNB,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.523263,0.004299,0.763301,0.005482,0.620877,0.003672,0.672957,0.003109,0.594223,0.005226
2,week5,max,RandomForestClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.508454,0.005570,0.525002,0.003861,0.516581,0.003741,0.595677,0.006471,0.572273,0.005008
3,week5,max,SVC,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.604104,0.005402,0.464120,0.009698,0.524876,0.005905,0.658852,0.003176,0.634305,0.002962
4,week5,max,KNeighborsClassifier,27244,struggling,"total_click_until_cutoff_max_strategy,active_d...",0.528010,0.010540,0.475166,0.010483,0.500191,0.010403,0.606032,0.007638,0.586661,0.008317
5,week5,sum,DecisionTreeClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.502443,0.008023,0.467663,0.008370,0.484381,0.006201,0.564044,0.008037,0.566620,0.006482
6,week5,sum,GaussianNB,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.521574,0.004439,0.771312,0.006357,0.622311,0.004280,0.673505,0.003113,0.592461,0.005516
7,week5,sum,RandomForestClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.509510,0.006247,0.526520,0.004350,0.517860,0.004480,0.591673,0.005117,0.573227,0.005525
8,week5,sum,SVC,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.604740,0.004937,0.468842,0.009892,0.528124,0.005821,0.658785,0.002916,0.635369,0.002643
9,week5,sum,KNeighborsClassifier,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.531268,0.003876,0.482840,0.006847,0.505882,0.004856,0.606051,0.003411,0.589451,0.003040


## 6. 결과 저장

전체 결과는 `paper_replication_baseline_results.csv`에 저장한다. cutoff별 대표 결과는 struggling 클래스 기준 `f1_mean`을 우선 기준으로 선택하고, 동률일 경우 `roc_auc_mean`, `recall_mean`, `accuracy_mean` 순서로 정렬해 `paper_replication_baseline_best_by_cutoff.csv`에 저장한다.

In [8]:
results_path = REPORT_TABLE_DIR / "paper_replication_baseline_results.csv"
best_path = REPORT_TABLE_DIR / "paper_replication_baseline_best_by_cutoff.csv"

sort_columns = ["cutoff", "f1_mean", "roc_auc_mean", "recall_mean", "accuracy_mean"]
best_by_cutoff = (
    results.sort_values(sort_columns, ascending=[True, False, False, False, False])
    .groupby("cutoff", as_index=False)
    .head(1)
    .sort_values("cutoff")
    .reset_index(drop=True)
)

results.to_csv(results_path, index=False, encoding="utf-8")
best_by_cutoff.to_csv(best_path, index=False, encoding="utf-8")

print(f"Saved full results: {results_path}")
print(f"Saved best by cutoff: {best_path}")

best_by_cutoff

Saved full results: C:\Users\osca0\Dev\studyDataMining\teamproject\reports\tables\paper_replication_baseline_results.csv
Saved best by cutoff: C:\Users\osca0\Dev\studyDataMining\teamproject\reports\tables\paper_replication_baseline_best_by_cutoff.csv


,cutoff,strategy,model,n_rows,positive_class,features,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std
0,week10,sum,GaussianNB,26069,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.516693,0.005510,0.771622,0.008029,0.618923,0.005781,0.704954,0.007003,0.610572,0.006680
1,week5,sum,GaussianNB,27244,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.521574,0.004439,0.771312,0.006357,0.622311,0.004280,0.673505,0.003113,0.592461,0.005516
2,week7,sum,GaussianNB,26776,struggling,"total_click_until_cutoff_sum_strategy,active_d...",0.521570,0.004371,0.771311,0.006753,0.622307,0.004255,0.688353,0.002799,0.601696,0.005372


## 7. 해석 시 주의사항

이 노트북은 엄격한 두-feature 논문 재현형 베이스라인이다. 성능이 낮거나 특정 모델이 불안정하더라도 이는 더 많은 특징을 추가하기 전의 기준선으로 해석해야 한다.

특히 이 실험은 인구통계, assessment, module/presentation, activity_type-specific click feature를 의도적으로 제외했기 때문에 최종 모델 후보가 아니다. VLE 중복 처리 문제의 영향은 max strategy와 sum strategy의 결과 차이로 별도 확인한다.